# LLM Pre-Training and Fine-Tuning

This notebook teaches the two-stage recipe behind every modern LLM: **pre-training** and **fine-tuning**. We build a miniature GPT-style transformer, pre-train it with next-token prediction, then compare three fine-tuning strategies inspired by Uber's production LLM work:

1. **Pre-training** — self-supervised language modeling on a general corpus
2. **Full fine-tuning** — update every parameter for a downstream task
3. **LoRA (Low-Rank Adaptation)** — update only a tiny set of injected weight matrices
4. **Frozen backbone** — only train a task head, the most extreme form of PEFT

All experiments run on CPU/MPS and are designed to finish in minutes. The goal is intuition, not benchmark scores.

---
### Why does this matter? (Uber's framing)
Uber found that **full fine-tuning beats LoRA on loss**, but LoRA uses far fewer GPUs and trains much faster. The right choice depends on your hardware budget and how much domain shift you need to cover. This notebook lets you observe that tradeoff directly.

## 0. Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math, copy, time
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

device = (
    torch.device('cuda') if torch.cuda.is_available()
    else torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cpu')
)
print(f'Device: {device}')

try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('default')

Device: mps


---
## Part 1 — What is Pre-Training?

### The core idea

Pre-training is **self-supervised learning on a massive unlabelled corpus**. The task is simple:

> Given tokens `[t₁, t₂, …, tₙ]`, predict `tₙ₊₁`.

There are no human labels. The labels are the text itself, shifted by one position. This is called **causal language modelling** (CLM) or **next-token prediction**.

```
Input:   The cat sat on the
Target:  cat sat on the mat
```

### Why does predicting the next word teach anything useful?

To predict well, the model must learn:
- Grammar (syntax)
- Facts (semantics)
- Reasoning patterns (pragmatics)

These compressed representations are the "pre-trained weights" that transfer to downstream tasks.

### Scale is everything
Real LLMs (GPT-4, Llama) are pre-trained on trillions of tokens with billions of parameters. We'll use a toy model, but the mechanics are identical.

## 1.1 — Build a Miniature GPT Transformer

Our model has the same architecture as GPT-2/Llama at a fraction of the scale:
- Token embeddings + positional embeddings
- N × Transformer blocks (causal self-attention + MLP)
- Linear output head → vocabulary logits

**Causal masking** ensures position `i` can only attend to positions `≤ i`, which is what makes next-token prediction valid during training.

In [2]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # reshape for multi-head attention
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        scale = math.sqrt(self.head_dim)
        attn = (q @ k.transpose(-2, -1)) / scale          # (B, H, T, T)

        # causal mask: upper triangle = -inf so future tokens are invisible
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        attn = attn.masked_fill(mask, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio * d_model),
            nn.GELU(),
            nn.Linear(mlp_ratio * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # pre-norm residual
        x = x + self.mlp(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=4,
                 max_seq_len=128, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        # weight tying: share embedding weights with output head
        self.head.weight = self.tok_emb.weight

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)   # (B, T, vocab_size)

    def count_params(self):
        return sum(p.numel() for p in self.parameters())

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            logits = self(idx[:, -128:])[:, -1, :] / temperature
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx


# Instantiate and inspect
VOCAB_SIZE = 65  # character-level (like GPT's original char demo)
model_cfg = dict(vocab_size=VOCAB_SIZE, d_model=128, n_heads=4, n_layers=4, max_seq_len=128)
test_model = MiniGPT(**model_cfg)
print(f'MiniGPT parameters: {test_model.count_params():,}')
print(f'Breakdown:')
for name, p in test_model.named_parameters():
    print(f'  {name:<40s} {tuple(p.shape)}')

MiniGPT parameters: 816,000
Breakdown:
  tok_emb.weight                           (65, 128)
  pos_emb.weight                           (128, 128)
  blocks.0.ln1.weight                      (128,)
  blocks.0.ln1.bias                        (128,)
  blocks.0.attn.qkv.weight                 (384, 128)
  blocks.0.attn.proj.weight                (128, 128)
  blocks.0.ln2.weight                      (128,)
  blocks.0.ln2.bias                        (128,)
  blocks.0.mlp.0.weight                    (512, 128)
  blocks.0.mlp.0.bias                      (512,)
  blocks.0.mlp.2.weight                    (128, 512)
  blocks.0.mlp.2.bias                      (128,)
  blocks.1.ln1.weight                      (128,)
  blocks.1.ln1.bias                        (128,)
  blocks.1.attn.qkv.weight                 (384, 128)
  blocks.1.attn.proj.weight                (128, 128)
  blocks.1.ln2.weight                      (128,)
  blocks.1.ln2.bias                        (128,)
  blocks.1.mlp.0.weight       

## 1.2 — Pre-Training Data: Two Corpora

We use **character-level tokenisation** to keep things self-contained (no HuggingFace tokeniser needed).

We prepare two corpora:
- **General corpus** — mix of prose, science text, general English
- **Domain corpus** — technical ML/AI text (simulates fine-tuning target domain)

Pre-training uses the general corpus. Fine-tuning adapts to the domain corpus.

In [3]:
GENERAL_TEXT = """
Once upon a time in a land far away, there lived a wise old king who ruled his kingdom with great
justice and mercy. The people loved him dearly, for he always listened to their needs and worked
tirelessly to ensure their happiness and prosperity.

The sun rises in the east and sets in the west. Rivers flow from mountains to the sea. Trees grow
tall in the forest, and flowers bloom in the meadows each spring. Birds sing in the morning, and
crickets chirp at night.

Science has transformed our understanding of the natural world. Through careful observation and
experimentation, researchers have uncovered the fundamental laws that govern matter, energy, space,
and time. From the smallest atom to the largest galaxy, nature follows elegant mathematical rules.

The history of civilization spans thousands of years. Ancient peoples built cities, developed writing
systems, and created art that endures to this day. Trade routes connected distant cultures, spreading
ideas, technology, and goods across continents.

In the realm of philosophy, great thinkers have pondered the nature of existence, knowledge, ethics,
and beauty. These questions remain alive and debated in every generation, for they touch on the
deepest concerns of human life.

Music has the power to move us emotionally and to connect people across languages and cultures.
Whether classical symphonies or folk songs passed down through generations, melody and rhythm
are universal forms of human expression.
"""

DOMAIN_TEXT = """
A neural network is a computational model inspired by the structure of the brain. It consists of
layers of interconnected neurons, each performing a weighted sum followed by a nonlinear activation.
Training adjusts the weights to minimise a loss function via gradient descent.

Pre-training is the first stage of modern large language model development. The model learns a
general representation of language by predicting the next token in a sequence. This self-supervised
objective requires no human annotation and scales with data and compute.

Fine-tuning adapts a pre-trained model to a specific downstream task. Full fine-tuning updates every
parameter, while parameter-efficient methods such as LoRA inject small trainable matrices into each
attention layer without modifying the frozen backbone.

LoRA works by decomposing a weight update into two low-rank matrices: delta W equals A times B,
where A has shape d times r and B has shape r times d, with r much smaller than d. Only A and B
are trained, reducing the number of trainable parameters by orders of magnitude.

The transformer architecture relies on self-attention to relate every token to every other token
in the sequence. Causal masking ensures that during language modelling the model cannot attend
to future tokens, preserving the autoregressive property needed for generation.

Gradient checkpointing trades compute for memory by recomputing activations during the backward
pass rather than storing them. Flash attention fuses the attention kernel to reduce memory bandwidth.
DeepSpeed ZeRO partitions optimizer states, gradients, and parameters across devices.

Evaluation of language models uses perplexity, which equals the exponent of the cross-entropy loss.
Lower perplexity indicates the model assigns higher probability to held-out text, meaning it has
learned a better distribution over the language.
"""

# Build shared vocabulary from both corpora
all_text = GENERAL_TEXT + DOMAIN_TEXT
chars = sorted(set(all_text))
VOCAB_SIZE = len(chars)
c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for c, i in c2i.items()}

encode = lambda s: [c2i[c] for c in s if c in c2i]
decode = lambda ids: ''.join(i2c[i] for i in ids)

print(f'Vocabulary size: {VOCAB_SIZE}')
print(f'General corpus tokens: {len(encode(GENERAL_TEXT)):,}')
print(f'Domain corpus tokens:  {len(encode(DOMAIN_TEXT)):,}')

Vocabulary size: 49
General corpus tokens: 1,481
Domain corpus tokens:  1,883


In [4]:
class TextDataset(Dataset):
    def __init__(self, text, seq_len=64):
        self.data = torch.tensor(encode(text), dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, i):
        chunk = self.data[i: i + self.seq_len + 1]
        return chunk[:-1], chunk[1:]   # input, target (shifted by 1)


SEQ_LEN = 64

# 80/20 split for general corpus
gen_ids = encode(GENERAL_TEXT)
split = int(0.8 * len(gen_ids))
gen_train_text = decode(gen_ids[:split])
gen_val_text   = decode(gen_ids[split:])

pretrain_train_ds = TextDataset(gen_train_text, SEQ_LEN)
pretrain_val_ds   = TextDataset(gen_val_text,   SEQ_LEN)

# Domain corpus for fine-tuning
dom_ids = encode(DOMAIN_TEXT)
dom_split = int(0.8 * len(dom_ids))
dom_train_text = decode(dom_ids[:dom_split])
dom_val_text   = decode(dom_ids[dom_split:])

finetune_train_ds = TextDataset(dom_train_text, SEQ_LEN)
finetune_val_ds   = TextDataset(dom_val_text,   SEQ_LEN)

def make_loader(ds, batch_size=32):
    return DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=True)

print(f'Pre-train train batches : {len(make_loader(pretrain_train_ds))}')
print(f'Fine-tune train batches : {len(make_loader(finetune_train_ds))}')

Pre-train train batches : 35
Fine-tune train batches : 45


## 1.3 — Pre-Training Loop

We train with the standard **cross-entropy loss** on the shifted token sequence. The loss tells us how many bits the model needs to encode each token — lower is better.

**Perplexity** is the exponent of the average cross-entropy loss:

$$\text{PPL} = e^{\mathcal{L}}$$

A perplexity of 10 means the model is as confused as if it had to choose uniformly among 10 equally likely options at each step.

In [5]:
def run_epoch(model, loader, optimizer=None, max_grad_norm=1.0):
    """One training or evaluation epoch. Returns average loss."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, n = 0.0, 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)                              # (B, T, V)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), y.view(-1)
            )
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()
            total_loss += loss.item()
            n += 1
    return total_loss / n


def train(model, train_loader, val_loader, epochs, lr=3e-3, label=''):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    history = {'train': [], 'val': []}
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        tr = run_epoch(model, train_loader, optimizer)
        vl = run_epoch(model, val_loader)
        scheduler.step()
        history['train'].append(tr)
        history['val'].append(vl)
        if epoch % max(1, epochs // 5) == 0 or epoch == 1:
            print(f'[{label}] epoch {epoch:3d}/{epochs}  '
                  f'train_loss={tr:.3f}  val_loss={vl:.3f}  '
                  f'val_ppl={math.exp(vl):.1f}  '
                  f'elapsed={time.time()-t0:.0f}s')
    return history


# --- Pre-train ---
model_cfg = dict(vocab_size=VOCAB_SIZE, d_model=128, n_heads=4,
                 n_layers=4, max_seq_len=128, dropout=0.1)

pretrained_model = MiniGPT(**model_cfg).to(device)
print(f'Parameters: {pretrained_model.count_params():,}')

pretrain_history = train(
    pretrained_model,
    make_loader(pretrain_train_ds),
    make_loader(pretrain_val_ds),
    epochs=60,
    lr=3e-3,
    label='PRE-TRAIN'
)

Parameters: 813,952
[PRE-TRAIN] epoch   1/60  train_loss=3.088  val_loss=2.996  val_ppl=20.0  elapsed=1s
[PRE-TRAIN] epoch  12/60  train_loss=0.412  val_loss=5.370  val_ppl=214.8  elapsed=5s
[PRE-TRAIN] epoch  24/60  train_loss=0.143  val_loss=6.281  val_ppl=534.6  elapsed=10s
[PRE-TRAIN] epoch  36/60  train_loss=0.100  val_loss=6.621  val_ppl=751.0  elapsed=15s
[PRE-TRAIN] epoch  48/60  train_loss=0.084  val_loss=6.861  val_ppl=954.5  elapsed=19s
[PRE-TRAIN] epoch  60/60  train_loss=0.079  val_loss=6.885  val_ppl=977.2  elapsed=24s


In [6]:
# Save the pre-trained weights — we'll reload these for each fine-tuning experiment
pretrained_state = copy.deepcopy(pretrained_model.state_dict())

# Sample text from the pre-trained model
prompt = "The history of"
idx = torch.tensor([encode(prompt)], device=device)
out = pretrained_model.generate(idx, max_new_tokens=120, temperature=0.8, top_k=10)
print('Pre-trained model generation:')
print('>>> ' + decode(out[0].tolist()))

Pre-trained model generation:
>>> The history of civilization spans thousands of years. Ancient peoples builes, builoplt built builes, built built built built cilt buit


---
## Part 2 — What is Fine-Tuning?

### The two-stage recipe

```
Stage 1 — Pre-training (expensive, done once)
  Large general corpus ──► Model learns language representations

Stage 2 — Fine-tuning (cheap, done per task)
  Small domain corpus ──► Model adapts to specific style / knowledge
```

### Why not just train on the domain corpus from scratch?

1. Domain corpora are **small** — you'd overfit immediately.
2. General language understanding (syntax, reasoning) is **expensive to learn** and **transfers well**.
3. Pre-training on general data, then fine-tuning on domain data, is almost always better than training from scratch on domain data alone.

### Three fine-tuning strategies

| Strategy | Parameters updated | GPU memory | Speed |
|---|---|---|---|
| **Full fine-tuning** | 100% | High | Slow |
| **LoRA** | ~1–5% | Low | Fast |
| **Frozen backbone** | <1% (head only) | Lowest | Fastest |

## 2.1 — Strategy A: Full Fine-Tuning

Every parameter in the network is updated. This gives the maximum flexibility to adapt but:
- Requires storing full gradients and optimizer states for all parameters
- Risk of **catastrophic forgetting** — the model may lose general capabilities
- Uber observed this gives the **lowest loss** compared to LoRA, but at high GPU cost

In [7]:
# Start from the pre-trained weights
full_ft_model = MiniGPT(**model_cfg).to(device)
full_ft_model.load_state_dict(pretrained_state)

# Count trainable parameters — should be 100%
trainable = sum(p.numel() for p in full_ft_model.parameters() if p.requires_grad)
total = full_ft_model.count_params()
print(f'Full fine-tuning — trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

full_ft_history = train(
    full_ft_model,
    make_loader(finetune_train_ds),
    make_loader(finetune_val_ds),
    epochs=40,
    lr=5e-4,   # lower lr than pre-training to avoid catastrophic forgetting
    label='FULL-FT'
)

Full fine-tuning — trainable: 813,952 / 813,952 (100.0%)
[FULL-FT] epoch   1/40  train_loss=3.174  val_loss=2.856  val_ppl=17.4  elapsed=1s
[FULL-FT] epoch   8/40  train_loss=0.220  val_loss=5.099  val_ppl=163.9  elapsed=4s
[FULL-FT] epoch  16/40  train_loss=0.142  val_loss=5.622  val_ppl=276.4  elapsed=8s
[FULL-FT] epoch  24/40  train_loss=0.119  val_loss=5.781  val_ppl=324.0  elapsed=12s
[FULL-FT] epoch  32/40  train_loss=0.111  val_loss=5.849  val_ppl=346.7  elapsed=16s
[FULL-FT] epoch  40/40  train_loss=0.108  val_loss=5.873  val_ppl=355.2  elapsed=20s


## 2.2 — Strategy B: LoRA (Low-Rank Adaptation)

### The key insight

For a pre-trained weight matrix **W** ∈ ℝ^(d×d), fine-tuning computes **W + ΔW**. LoRA hypothesises that **ΔW has low intrinsic rank** — it lives in a much smaller subspace.

Instead of learning **ΔW** directly (d² parameters), LoRA learns:

$$\Delta W = BA \quad \text{where } B \in \mathbb{R}^{d \times r},\; A \in \mathbb{R}^{r \times d},\; r \ll d$$

Parameters reduced from **d²** to **2dr**. For d=1024, r=8 that is 2M → 16K, a **128× reduction**.

### During inference

The LoRA matrices can be **merged** back into W:
```
W_eff = W + (B @ A) * scale
```
Zero inference overhead. The frozen weights are never changed.

### Where to inject LoRA?

Original LoRA paper injects into Q and V projections of each attention layer. We'll inject into Q, K, V, and the output projection for completeness.

In [8]:
class LoRALinear(nn.Module):
    """Wraps a frozen Linear layer and adds a low-rank trainable branch."""

    def __init__(self, linear: nn.Linear, rank: int = 8, alpha: float = 16.0):
        super().__init__()
        self.linear = linear
        self.linear.weight.requires_grad_(False)   # freeze original weights
        if self.linear.bias is not None:
            self.linear.bias.requires_grad_(False)

        d_out, d_in = linear.weight.shape
        self.rank = rank
        # scale factor: alpha/rank keeps the output magnitude stable as rank changes
        self.scale = alpha / rank

        # A is initialised with random Gaussian, B with zeros
        # This means at the start delta_W = B @ A = 0, so training begins
        # from exactly the pre-trained checkpoint.
        self.A = nn.Parameter(torch.randn(rank, d_in) * 0.01)
        self.B = nn.Parameter(torch.zeros(d_out, rank))

    def forward(self, x):
        # Original frozen path + low-rank adaptation
        return self.linear(x) + (x @ self.A.T @ self.B.T) * self.scale

    def merge(self) -> nn.Linear:
        """Absorb LoRA into the base weight (for inference). Returns a plain Linear."""
        merged = copy.deepcopy(self.linear)
        merged.weight.data += (self.B @ self.A) * self.scale
        merged.weight.requires_grad_(True)
        return merged


def inject_lora(model: MiniGPT, rank: int = 8, alpha: float = 16.0) -> MiniGPT:
    """Replace QKV and projection linears in each attention block with LoRALinear."""
    for block in model.blocks:
        # The CausalSelfAttention has a fused qkv Linear(d, 3d)
        # and a projection Linear(d, d). We LoRA-ify both.
        block.attn.qkv  = LoRALinear(block.attn.qkv,  rank, alpha)
        block.attn.proj = LoRALinear(block.attn.proj, rank, alpha)
    return model


# Build LoRA model: load pre-trained weights, then inject LoRA
lora_model = MiniGPT(**model_cfg).to(device)
lora_model.load_state_dict(pretrained_state)
lora_model = inject_lora(lora_model, rank=8, alpha=16.0)

# Freeze everything except LoRA parameters
for name, p in lora_model.named_parameters():
    if 'lora' not in name.lower() and '.A' not in name and '.B' not in name:
        p.requires_grad_(False)

trainable_lora = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
total_lora     = sum(p.numel() for p in lora_model.parameters())
print(f'LoRA — trainable: {trainable_lora:,} / {total_lora:,} ({100*trainable_lora/total_lora:.2f}%)')
print(f'Parameter reduction vs full fine-tuning: {total_lora/trainable_lora:.1f}x')

LoRA — trainable: 24,576 / 838,528 (2.93%)
Parameter reduction vs full fine-tuning: 34.1x


In [9]:
# Train only the LoRA parameters
lora_history = train(
    lora_model,
    make_loader(finetune_train_ds),
    make_loader(finetune_val_ds),
    epochs=40,
    lr=1e-3,   # LoRA benefits from slightly higher lr since fewer parameters update
    label='LORA'
)

RuntimeError: Tensor for argument #2 'mat2' is on CPU, but expected it to be on GPU (while checking arguments for mm)

## 2.3 — Strategy C: Frozen Backbone (Linear Probe)

The most extreme form of parameter-efficient fine-tuning: **freeze the entire backbone** and only train the output head. This treats the pre-trained model as a fixed feature extractor.

- Used as a baseline to measure how much general representations already capture domain knowledge
- Often surprisingly competitive for classification tasks
- For language modelling, the head **is** the vocabulary projection — same linear layer

In [ ]:
frozen_model = MiniGPT(**model_cfg).to(device)
frozen_model.load_state_dict(pretrained_state)

# Freeze everything except the final head
for name, p in frozen_model.named_parameters():
    if 'head' not in name:
        p.requires_grad_(False)

trainable_frozen = sum(p.numel() for p in frozen_model.parameters() if p.requires_grad)
print(f'Frozen backbone — trainable: {trainable_frozen:,} / {frozen_model.count_params():,} '
      f'({100*trainable_frozen/frozen_model.count_params():.2f}%)')

frozen_history = train(
    frozen_model,
    make_loader(finetune_train_ds),
    make_loader(finetune_val_ds),
    epochs=40,
    lr=1e-3,
    label='FROZEN'
)

---
## Part 3 — Analysis and Comparison

In [ ]:
# ── 3.1 Training curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

experiments = [
    ('Full fine-tuning', full_ft_history,  'steelblue'),
    ('LoRA',            lora_history,      'darkorange'),
    ('Frozen backbone', frozen_history,    'seagreen'),
]

for label, h, color in experiments:
    axes[0].plot(h['train'], label=label, color=color, lw=2)
    axes[1].plot(h['val'],   label=label, color=color, lw=2)

axes[0].set(title='Train Loss (domain corpus)', xlabel='Epoch', ylabel='Cross-entropy loss')
axes[1].set(title='Validation Loss (domain corpus)', xlabel='Epoch', ylabel='Cross-entropy loss')

for ax in axes:
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Fine-tuning Strategy Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Final results table
print('\n Final validation loss on domain corpus:')
print(f'  Full fine-tuning : {full_ft_history["val"][-1]:.4f}  (ppl={math.exp(full_ft_history["val"][-1]):.1f})')
print(f'  LoRA             : {lora_history["val"][-1]:.4f}  (ppl={math.exp(lora_history["val"][-1]):.1f})')
print(f'  Frozen backbone  : {frozen_history["val"][-1]:.4f}  (ppl={math.exp(frozen_history["val"][-1]):.1f})')

## 3.2 — Parameter Efficiency vs. Performance

This plot visualises the classic tradeoff: how much does each strategy achieve, and at what parameter cost?

In [ ]:
strategies = {
    'Full\nfine-tuning': {
        'pct_params': 100.0,
        'val_loss': full_ft_history['val'][-1],
        'color': 'steelblue'
    },
    'LoRA\n(r=8)': {
        'pct_params': 100 * trainable_lora / total_lora,
        'val_loss': lora_history['val'][-1],
        'color': 'darkorange'
    },
    'Frozen\nbackbone': {
        'pct_params': 100 * trainable_frozen / frozen_model.count_params(),
        'val_loss': frozen_history['val'][-1],
        'color': 'seagreen'
    },
}

fig, ax = plt.subplots(figsize=(8, 6))

for name, info in strategies.items():
    ax.scatter(info['pct_params'], info['val_loss'],
               s=300, color=info['color'], zorder=5, edgecolors='k', linewidths=1.2)
    ax.annotate(name, (info['pct_params'], info['val_loss']),
                textcoords='offset points', xytext=(8, 4), fontsize=11,
                color=info['color'], fontweight='bold')

ax.set_xlabel('% Trainable Parameters', fontsize=12)
ax.set_ylabel('Validation Loss (lower = better)', fontsize=12)
ax.set_title('Parameter Efficiency vs. Performance\n(Uber-style tradeoff view)', fontsize=13)
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
ax.invert_yaxis()

# Annotate with "ideal" direction
ax.annotate('← Fewer params (cheaper)\n↑ Better performance (ideal corner: top-left)',
             xy=(0.02, 0.95), xycoords='axes fraction', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

## 3.3 — LoRA: Understanding the Low-Rank Decomposition

Let's visualise the weight updates learned by LoRA to build intuition for *why* low-rank works.

In [ ]:
# Extract the LoRA weight update for the first block's QKV projection
first_lora = lora_model.blocks[0].attn.qkv
A = first_lora.A.detach().cpu()   # (r, d_in)
B = first_lora.B.detach().cpu()   # (d_out, r)
delta_W = (B @ A) * first_lora.scale  # (d_out, d_in)

# Singular value decomposition of delta_W reveals its rank structure
U, S, Vh = torch.linalg.svd(delta_W, full_matrices=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Visualise delta_W
im = axes[0].imshow(delta_W.numpy(), aspect='auto', cmap='RdBu', vmin=-0.1, vmax=0.1)
plt.colorbar(im, ax=axes[0])
axes[0].set_title(f'ΔW = B @ A  (shape {tuple(delta_W.shape)})', fontsize=11)
axes[0].set_xlabel('Input dimension')
axes[0].set_ylabel('Output dimension')

# 2. Singular values — they fall off quickly showing low-rank structure
axes[1].bar(range(len(S)), S.numpy(), color='steelblue', edgecolor='k', linewidth=0.5)
axes[1].set_title('Singular values of ΔW\n(fast decay = low-rank structure)', fontsize=11)
axes[1].set_xlabel('Singular value index')
axes[1].set_ylabel('Magnitude')
axes[1].grid(True, alpha=0.3)

# 3. Cumulative explained variance
cum_var = torch.cumsum(S**2, dim=0) / (S**2).sum()
axes[2].plot(cum_var.numpy() * 100, 'o-', color='darkorange', markersize=5)
axes[2].axhline(95, color='red', linestyle='--', label='95% threshold')
axes[2].set_title('Cumulative variance explained by\ntop-k singular values', fontsize=11)
axes[2].set_xlabel('k (number of components)')
axes[2].set_ylabel('Variance explained (%)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

rank_for_95 = int((cum_var >= 0.95).nonzero()[0].item()) + 1
print(f'LoRA rank used: {A.shape[0]}')
print(f'Singular values of ΔW: {S.numpy().round(4)}')
print(f'Components needed to explain 95% of variance: {rank_for_95}')
print(f'→ The weight update is indeed low-rank, validating the LoRA hypothesis.')

plt.tight_layout()
plt.show()

## 3.4 — Qualitative Generation Comparison

Does each model capture domain-specific language after fine-tuning?

In [ ]:
prompts = [
    "Fine-tuning adapts",
    "The transformer",
    "Gradient",
]

models_to_compare = [
    ('Pre-trained (no fine-tuning)', pretrained_model),
    ('Full fine-tuning',            full_ft_model),
    ('LoRA',                        lora_model),
    ('Frozen backbone',             frozen_model),
]

for prompt in prompts:
    print('=' * 70)
    print(f'Prompt: "{prompt}"')
    print('=' * 70)
    idx = torch.tensor([encode(prompt)], device=device)
    for name, m in models_to_compare:
        out = m.generate(idx.clone(), max_new_tokens=80, temperature=0.7, top_k=10)
        generated = decode(out[0].tolist())
        print(f'  [{name}]')
        print(f'  {generated}')
        print()

## 3.5 — LoRA Rank Ablation: How Much Rank Do You Need?

LoRA's `r` (rank) controls the capacity of the adaptation. Higher rank = more parameters = potentially better fit, but diminishing returns.

In [ ]:
ranks = [1, 2, 4, 8, 16]
rank_results = {}

for r in ranks:
    m = MiniGPT(**model_cfg).to(device)
    m.load_state_dict(pretrained_state)
    m = inject_lora(m, rank=r, alpha=float(r))  # alpha = rank is a common heuristic
    for name, p in m.named_parameters():
        if '.A' not in name and '.B' not in name:
            p.requires_grad_(False)

    n_trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    h = train(m, make_loader(finetune_train_ds), make_loader(finetune_val_ds),
              epochs=40, lr=1e-3, label=f'LoRA r={r}')
    rank_results[r] = {'val_loss': h['val'][-1], 'n_params': n_trainable}

# Visualise rank ablation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

rs = list(rank_results.keys())
losses = [rank_results[r]['val_loss'] for r in rs]
params = [rank_results[r]['n_params'] for r in rs]

axes[0].plot(rs, losses, 'o-', color='darkorange', markersize=8, lw=2)
axes[0].axhline(full_ft_history['val'][-1], color='steelblue',
                linestyle='--', label='Full fine-tuning')
axes[0].set(xlabel='LoRA rank (r)', ylabel='Val loss', title='LoRA Rank vs. Val Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(params, losses, 'o-', color='seagreen', markersize=8, lw=2)
for r, p, l in zip(rs, params, losses):
    axes[1].annotate(f'r={r}', (p, l), textcoords='offset points', xytext=(4, 4))
axes[1].axhline(full_ft_history['val'][-1], color='steelblue',
                linestyle='--', label='Full fine-tuning')
axes[1].set(xlabel='Trainable parameters', ylabel='Val loss',
            title='Trainable Params vs. Val Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('LoRA Rank Ablation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nRank ablation results:')
for r in rs:
    print(f'  r={r:2d} — params={rank_results[r]["n_params"]:5,} — '
          f'val_loss={rank_results[r]["val_loss"]:.4f}')

---
## Part 4 — Memory and Compute Intuition (Uber's Key Findings)

You can't directly measure GPU memory in this notebook, but we can count the **number of tensors that need optimizer states**. In AdamW, every trainable parameter needs 2 extra state tensors (first and second moment), so:

$$\text{Optimizer memory} \approx 3 \times |\theta_{\text{trainable}}| \times \text{bytes per param}$$

This is precisely why Uber found LoRA uses far fewer GPUs.

In [ ]:
def estimate_memory_mb(n_trainable, bytes_per_param=4):
    """Estimate optimizer state memory (params + m1 + m2 for AdamW)."""
    return 3 * n_trainable * bytes_per_param / 1e6

full_params  = total
lora_params  = trainable_lora
froz_params  = trainable_frozen

strategies_mem = {
    'Full fine-tuning': full_params,
    'LoRA (r=8)':       lora_params,
    'Frozen backbone':  froz_params,
}

print('Memory breakdown (relative to full fine-tuning):')
full_mem = estimate_memory_mb(full_params)
print(f'{"Strategy":<22} {"Trainable params":>18} {"Optimizer mem (MB)":>20} {"Relative":>10}')
print('-' * 74)
for name, p in strategies_mem.items():
    mem = estimate_memory_mb(p)
    print(f'{name:<22} {p:>18,} {mem:>20.1f} {mem/full_mem:>10.3f}x')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

names = list(strategies_mem.keys())
param_counts = [strategies_mem[n] for n in names]
mem_estimates = [estimate_memory_mb(strategies_mem[n]) for n in names]
colors = ['steelblue', 'darkorange', 'seagreen']

axes[0].bar(names, param_counts, color=colors, edgecolor='k', linewidth=0.8)
axes[0].set(ylabel='Trainable parameters', title='Trainable Parameter Count')
axes[0].set_yscale('log')
for i, v in enumerate(param_counts):
    axes[0].text(i, v * 1.3, f'{v:,}', ha='center', va='bottom', fontsize=9)

axes[1].bar(names, mem_estimates, color=colors, edgecolor='k', linewidth=0.8)
axes[1].set(ylabel='Optimizer state memory (MB, FP32)',
            title='Estimated Optimizer Memory')
axes[1].set_yscale('log')
for i, v in enumerate(mem_estimates):
    axes[1].text(i, v * 1.3, f'{v:.2f} MB', ha='center', va='bottom', fontsize=9)

plt.suptitle('Why LoRA Saves GPU Memory', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nAt fp16 (Llama 70B scale, ~70B params):')
scale = 70e9 / full_params
print(f'  Full fine-tuning optimizer states ≈ {estimate_memory_mb(70e9, 2)/1024:.0f} GB')
print(f'  LoRA optimizer states             ≈ {estimate_memory_mb(70e9 * (lora_params/full_params), 2)/1024:.1f} GB')
print(f'  → This is why Uber uses LoRA for 70B models on fewer GPUs.')

---
## Summary

### What we built
- A miniature GPT transformer with causal self-attention and pre-norm residual connections
- Pre-training on a general corpus with next-token prediction
- Three fine-tuning strategies: full, LoRA, and frozen backbone
- LoRA from scratch: weight decomposition into low-rank matrices B and A

### Key takeaways

| Concept | What to remember |
|---|---|
| **Pre-training** | Self-supervised next-token prediction on large unlabelled corpora. The loss is cross-entropy; the metric is perplexity. |
| **Fine-tuning** | Adapts frozen pre-trained representations to a domain with far less data and compute. |
| **Full fine-tuning** | Lowest loss but proportional GPU memory and compute cost. Risk of catastrophic forgetting. |
| **LoRA** | Injects low-rank matrices ΔW = BA into frozen weights. ~10–100× fewer trainable params, near-competitive loss. Zero inference overhead after merging. |
| **Frozen backbone** | Only the output head trains. Useful baseline; limited by fixed representations. |
| **Uber's finding** | For Llama 70B: full fine-tuning wins on loss, but LoRA is the practical default due to GPU memory constraints. The right rank `r` is task-dependent. |

### What to explore next
- **QLoRA**: quantise the frozen backbone to 4-bit (NF4) before injecting LoRA — reduces memory further
- **Instruction fine-tuning (SFT)**: fine-tune on `(instruction, response)` pairs with a supervised loss
- **RLHF / DPO**: align the model to human preferences after SFT
- **Flash Attention**: fused attention kernel that halves memory bandwidth with the same output
- **DeepSpeed ZeRO**: shard optimizer states, gradients, and weights across GPUs for truly large models